# Unit Test Generation — Autoresearch
PhD Research: Plain LLM vs Simple RAG vs Iterative Critique RAG for unit test generation.

**Steps:**
1. Mount Google Drive (persist everything across resets)
2. Install dependencies
3. Install & start Ollama, pull model
4. Clone repo
5. One-time setup (download dataset + build knowledge base)
6. Run single experiment
7. Run all 12 experiments (full PhD comparison)
8. Visualize results
9. Cross-task comparison (RQ4)

## Step 1: Mount Google Drive
**Run this first.** All results, logs, charts, and checkpoints are saved to Drive so nothing is lost on runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All PhD outputs go here — edit if you want a different Drive folder
DRIVE_DIR = '/content/drive/MyDrive/PhD_autoresearch'

# Sub-directories (created automatically)
LOGS_DIR        = os.path.join(DRIVE_DIR, 'logs')
PLOTS_DIR       = os.path.join(DRIVE_DIR, 'plots')
RESULTS_DIR     = os.path.join(DRIVE_DIR, 'results')
CHECKPOINTS_DIR = os.path.join(DRIVE_DIR, 'checkpoints')

for d in [LOGS_DIR, PLOTS_DIR, RESULTS_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. All outputs will be saved to:')
print(f'  Logs:        {LOGS_DIR}')
print(f'  Plots:       {PLOTS_DIR}')
print(f'  Results:     {RESULTS_DIR}')
print(f'  Checkpoints: {CHECKPOINTS_DIR}')

## Step 2: Install Python dependencies

In [ ]:
!pip install -q ollama datasets sentence-transformers scikit-learn nltk rouge beautifulsoup4 numpy pandas matplotlib

## Step 3: Install Ollama and pull model

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('Ollama server started.')

In [ ]:
# Choose your model based on available GPU RAM:
# T4  (15GB): llama3.2:latest, phi3:mini, qwen2.5:3b, deepseek-coder:6.7b
# A100(40GB): llama3.1:8b, phi4:14b, qwen2.5:14b

MODEL = 'llama3.2:latest'   # change this if needed
!ollama pull {MODEL}

## Step 4: Clone the repo

In [ ]:
REPO_DIR = '/content/autoresearch'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/balajivenky06/autoresearch.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!ls

## Step 5: One-time setup (download dataset + build knowledge base)
Only needs to run once per Colab session. The dataset/KB are cached in `~/.cache/autoresearch_unitest/`.

In [ ]:
# Downloads HumanEval + MBPP from HuggingFace (~2 min)
# Fetches pytest/unittest docs and encodes them (~2 min)
!python prepare_unitest.py

## Step 6: Configure and run a single experiment
Use `set_experiment()` to pick METHOD, REASONING, and model. Then run the next cell.

In [ ]:
import re, ast

VALID_METHODS    = {'plain_llm', 'simple_rag', 'iterative_critique'}
VALID_REASONINGS = {'base', 'cot', 'tot', 'got'}

def set_experiment(method='plain_llm', reasoning='base', model=MODEL):
    """Update train_unitest.py config in-place."""
    assert method in VALID_METHODS, f'method must be one of {VALID_METHODS}'
    assert reasoning in VALID_REASONINGS, f'reasoning must be one of {VALID_REASONINGS}'

    with open('train_unitest.py', 'r') as f:
        content = f.read()

    content = re.sub(r'^METHOD\s*=.*$',         'METHOD    = "' + method    + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^REASONING\s*=.*$',       'REASONING = "' + reasoning + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^GENERATOR_MODEL\s*=.*$', 'GENERATOR_MODEL = "' + model + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^HELPER_MODEL\s*=.*$',    'HELPER_MODEL    = "' + model + '"', content, flags=re.MULTILINE)

    try:
        ast.parse(content)
    except SyntaxError as e:
        raise RuntimeError('set_experiment produced invalid Python: ' + str(e))

    with open('train_unitest.py', 'w') as f:
        f.write(content)
    print(f'Config set: method={method}, reasoning={reasoning}, model={model}')

# Set your experiment here:
set_experiment(method='plain_llm', reasoning='base', model=MODEL)

In [ ]:
import shutil

# Run the experiment — log saved to Drive immediately
log_path = os.path.join(LOGS_DIR, 'single_run.log')
!python train_unitest.py 2>&1 | tee {log_path}

# Copy results TSV to Drive
if os.path.exists('results_unitest.tsv'):
    shutil.copy('results_unitest.tsv', os.path.join(RESULTS_DIR, 'results_unitest.tsv'))
    print(f'results_unitest.tsv saved to Drive.')

In [ ]:
# Quick summary of the last run
!grep -E '^val_score:|^method:|^model:|^avg_' {log_path}

## Step 7: Run all 12 experiments (full PhD comparison)
3 methods × 4 reasoning techniques. **~2–4 hours on A100.**

If the runtime resets mid-run, re-run Steps 1–5, then re-run this cell — it will resume from the last checkpoint.

In [ ]:
import subprocess, pandas as pd, shutil

EXPERIMENTS = [
    ('plain_llm',          'base'),
    ('plain_llm',          'cot'),
    ('plain_llm',          'tot'),
    ('plain_llm',          'got'),
    ('simple_rag',         'base'),
    ('simple_rag',         'cot'),
    ('simple_rag',         'tot'),
    ('simple_rag',         'got'),
    ('iterative_critique', 'base'),
    ('iterative_critique', 'cot'),
    ('iterative_critique', 'tot'),
    ('iterative_critique', 'got'),
]

results = []
sep = '=' * 55

for method, reasoning in EXPERIMENTS:
    print(f'\n{sep}')
    print(f'  Running: {method} / {reasoning}')
    print(sep)

    set_experiment(method=method, reasoning=reasoning, model=MODEL)

    log_path = os.path.join(LOGS_DIR, f'{method}_{reasoning}.log')
    try:
        ret = subprocess.run(
            ['python', 'train_unitest.py'],
            capture_output=True, text=True, timeout=900
        )
        output = ret.stdout + ret.stderr
        with open(log_path, 'w') as f:
            f.write(output)
        print(f'  Log saved: {log_path}')
    except subprocess.TimeoutExpired:
        print('  TIMEOUT — skipping')
        results.append({'method': f'{method}/{reasoning}', 'val_score': 0.0, 'status': 'timeout'})
        continue

    # Parse key metrics from stdout
    row = {'method': f'{method}/{reasoning}', 'model': MODEL, 'status': 'ok'}
    for line in output.splitlines():
        for key in ['val_score', 'avg_noise_rate', 'avg_faithfulness',
                    'avg_retrieval_secs', 'avg_llm_secs', 'avg_syntax_valid',
                    'avg_edge_coverage', 'avg_rouge_1']:
            if line.strip().startswith(key + ':'):
                try:
                    row[key] = float(line.split(':')[1].strip())
                except Exception:
                    pass
    if row.get('val_score', 0.0) == 0.0:
        row['status'] = 'crash'
    results.append(row)
    print(f'  val_score: {row.get("val_score", 0):.6f}  [{row["status"]}]')

    # Copy results_unitest.tsv to Drive after every experiment
    if os.path.exists('results_unitest.tsv'):
        shutil.copy('results_unitest.tsv', os.path.join(RESULTS_DIR, 'results_unitest.tsv'))

# Summary table
df = pd.DataFrame(results).sort_values('val_score', ascending=False)
print(f'\n{sep}')
print('  FINAL RESULTS')
print(sep)
print(df.to_string(index=False))

# Save summary to Drive
summary_path = os.path.join(RESULTS_DIR, 'summary_all_experiments.csv')
df.to_csv(summary_path, index=False)
print(f'\nSummary saved to Drive: {summary_path}')

## Step 8: Visualize results
Generates 7 KPI charts and saves them to Drive.

In [ ]:
import shutil
from IPython.display import Image, display

# Generate all charts into plots_unitest/
!python visualize_unitest.py

# Copy charts to Drive and display
LOCAL_PLOTS = 'plots_unitest'
charts = [
    'heatmap.png',
    'grouped_bar.png',
    'radar.png',
    'per_metric_bar.png',
    'noise_rate.png',
    'cost_breakdown.png',
    'faithfulness.png',
]

for fname in charts:
    src = os.path.join(LOCAL_PLOTS, fname)
    if os.path.exists(src):
        dst = os.path.join(PLOTS_DIR, fname)
        shutil.copy(src, dst)
        print(f'{fname} → saved to Drive')
        display(Image(src))
    else:
        print(f'{fname} — not generated (run Step 7 first)')

## Step 9: Cross-task comparison — RQ4
Requires `results_docstring.tsv` from the RAG-Docstring repo.
Run `extract_docstring_results.py` locally first, then upload `results_docstring.tsv` to Drive.

In [ ]:
import shutil

# Copy results_docstring.tsv from Drive into repo working dir (if available)
docstring_tsv_drive = os.path.join(RESULTS_DIR, 'results_docstring.tsv')
if os.path.exists(docstring_tsv_drive):
    shutil.copy(docstring_tsv_drive, 'results_docstring.tsv')
    print('results_docstring.tsv loaded from Drive.')
else:
    print(f'results_docstring.tsv not found in Drive ({docstring_tsv_drive}).')
    print('Upload it to Drive first, then re-run this cell.')

In [ ]:
from IPython.display import Image, display

# Generate cross-task comparison charts
!python compare_tasks.py

# Copy to Drive and display
compare_charts = [
    'faithfulness_by_task.png',
    'noise_vs_faithfulness.png',
    'pareto.png',
]
for fname in compare_charts:
    src = os.path.join('plots_compare', fname)
    if os.path.exists(src):
        dst = os.path.join(PLOTS_DIR, f'compare_{fname}')
        shutil.copy(src, dst)
        print(f'{fname} → saved to Drive')
        display(Image(src))

# Copy summary table to Drive
if os.path.exists('plots_compare/summary_table.tsv'):
    shutil.copy('plots_compare/summary_table.tsv',
                os.path.join(RESULTS_DIR, 'cross_task_summary.tsv'))
    print('cross_task_summary.tsv saved to Drive.')